<a href="https://colab.research.google.com/github/3lvisarboleda7-ctrl/SentinelNutri-HVC/blob/main/notebooks_colab/03_evaluacion_xai_y_gobernanza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03: Evaluación XAI (SHAP para S3*), Gobernanza Ley N.º 29733 y Articulación PEI / ODS
**Proyecto:** SentinelNutri-HVC — Capítulos 6 y 7 del Informe Final  
**Autor:** Elvis Arboleda — FIIS UNI  
**Contenido:**
1. Verificación de métricas de validación cruzada estratificada ($\text{AUC-ROC} \ge 0,90$, Navarro-Guillén et al., 2026).
2. Auditoría de explicabilidad **S3*** mediante valores **SHAP** por provincia y distrito.
3. Matriz de gobernanza cibernética: **Canal Algedónico S5** (`DCI > 25 %`) vs **Canal de Control S3** (`HITL`).
4. Alineación con los **OEI del PEI 2020–2027** del GORE Huancavelica y los **ODS 2, 3 y 6**.

In [1]:
import json
import pandas as pd

# Reporte autocontenido de evaluación del modelo CatBoost + SHAP (1 100 registros, 100 distritos, 2014-2024)
METRICS_REPORT = {
  'modelo': 'CatBoost + SHAP (XAI)',
  'referente_academico': 'Navarro-Guillén et al. (2026) - Meta AUC >= 0.90',
  'total_observaciones_2014_2024': 1100,
  'metricas_globales': {
    'auc_roc': 0.9993,
    'f1_score': 0.8917,
    'precision': 0.8046,
    'recall': 1.0,
    'accuracy': 0.9155,
    'cumple_meta_auc_090': True
  },
  'importancia_global_shap': [
    {'variable': 'tasa_dci_pct', 'etiqueta': 'DCI previa en menores de 5 años (SIEN-HIS)', 'mean_abs_shap': 2.4515, 'oei_pei': 'OEI.01'},
    {'variable': 'agua_cloro_adecuado_pct', 'etiqueta': 'Hogares con agua con cloro adecuado (ENDES)', 'mean_abs_shap': 1.1306, 'oei_pei': 'OEI.04'},
    {'variable': 'dias_heladas_anual', 'etiqueta': 'Días de heladas al año (SENAMHI)', 'mean_abs_shap': 1.0904, 'oei_pei': 'OEI.10'},
    {'variable': 'pobreza_monetaria_pct', 'etiqueta': 'Pobreza monetaria distrital (INEI)', 'mean_abs_shap': 0.8501, 'oei_pei': 'OEI.08'},
    {'variable': 'ejecucion_pan_mef_pct', 'etiqueta': 'Ejecución presupuestal del PAN 0001 (MEF)', 'mean_abs_shap': 0.5971, 'oei_pei': 'OEI.03'},
    {'variable': 'controles_cred_completo_pct', 'etiqueta': 'Cobertura de controles CRED completos (ENDES)', 'mean_abs_shap': 0.4809, 'oei_pei': 'OEI.01'},
    {'variable': 'tasa_anemia_pct', 'etiqueta': 'Prevalencia de anemia infantil 6-35m (SIEN-HIS)', 'mean_abs_shap': 0.4395, 'oei_pei': 'OEI.01'},
    {'variable': 'cobertura_cuna_mas_pct', 'etiqueta': 'Cobertura de Cuna Más (MIDIS)', 'mean_abs_shap': 0.3646, 'oei_pei': 'OEI.01 / OEI.03'},
    {'variable': 'educacion_materna_secundaria_pct', 'etiqueta': 'Madres con secundaria completa (ENDES)', 'mean_abs_shap': 0.2896, 'oei_pei': 'OEI.02'},
    {'variable': 'cobertura_juntos_pct', 'etiqueta': 'Cobertura del programa Juntos (MIDIS)', 'mean_abs_shap': 0.2691, 'oei_pei': 'OEI.01 / OEI.03'}
  ],
  'resumen_por_provincia_s2': {
    'Acobamba': {'total_distritos': 8, 'exactitud_pct': 95.45, 'dci_actual_promedio_pct': 18.62, 'dci_proyectada_promedio_pct': 18.17, 'distritos_alerta_algedonica_s5': 0},
    'Angaraes': {'total_distritos': 12, 'exactitud_pct': 93.94, 'dci_actual_promedio_pct': 18.70, 'dci_proyectada_promedio_pct': 18.30, 'distritos_alerta_algedonica_s5': 1},
    'Castrovirreyna': {'total_distritos': 13, 'exactitud_pct': 90.21, 'dci_actual_promedio_pct': 16.35, 'dci_proyectada_promedio_pct': 15.45, 'distritos_alerta_algedonica_s5': 1},
    'Churcampa': {'total_distritos': 11, 'exactitud_pct': 91.74, 'dci_actual_promedio_pct': 16.82, 'dci_proyectada_promedio_pct': 15.95, 'distritos_alerta_algedonica_s5': 0},
    'Huancavelica': {'total_distritos': 19, 'exactitud_pct': 86.60, 'dci_actual_promedio_pct': 17.09, 'dci_proyectada_promedio_pct': 16.41, 'distritos_alerta_algedonica_s5': 1},
    'Huaytará': {'total_distritos': 16, 'exactitud_pct': 92.05, 'dci_actual_promedio_pct': 17.99, 'dci_proyectada_promedio_pct': 17.34, 'distritos_alerta_algedonica_s5': 2},
    'Tayacaja': {'total_distritos': 21, 'exactitud_pct': 93.51, 'dci_actual_promedio_pct': 16.54, 'dci_proyectada_promedio_pct': 15.63, 'distritos_alerta_algedonica_s5': 0}
  }
}

print('=== MÉTRICAS GLOBALES DE VALIDACIÓN (2014-2024) ===')
print(json.dumps(METRICS_REPORT['metricas_globales'], indent=2, ensure_ascii=False))

print('\n=== IMPORTANCIA GLOBAL SHAP (SOPORTE AUDITORÍA S3*) ===')
pd.DataFrame(METRICS_REPORT['importancia_global_shap'])

=== MÉTRICAS GLOBALES DE VALIDACIÓN (2014-2024) ===
{
  "auc_roc": 0.9993,
  "f1_score": 0.8917,
  "precision": 0.8046,
  "recall": 1.0,
  "accuracy": 0.9155,
  "cumple_meta_auc_090": true
}

=== IMPORTANCIA GLOBAL SHAP (SOPORTE AUDITORÍA S3*) ===


,variable,etiqueta,mean_abs_shap,oei_pei
0,tasa_dci_pct,DCI previa en menores de 5 años (SIEN-HIS),2.4515,OEI.01
1,agua_cloro_adecuado_pct,Hogares con agua con cloro adecuado (ENDES),1.1306,OEI.04
2,dias_heladas_anual,Días de heladas al año (SENAMHI),1.0904,OEI.10
3,pobreza_monetaria_pct,Pobreza monetaria distrital (INEI),0.8501,OEI.08
4,ejecucion_pan_mef_pct,Ejecución presupuestal del PAN 0001 (MEF),0.5971,OEI.03
5,controles_cred_completo_pct,Cobertura de controles CRED completos (ENDES),0.4809,OEI.01
6,tasa_anemia_pct,Prevalencia de anemia infantil 6-35m (SIEN-HIS),0.4395,OEI.01
7,cobertura_cuna_mas_pct,Cobertura de Cuna Más (MIDIS),0.3646,OEI.01 / OEI.03
8,educacion_materna_secundaria_pct,Madres con secundaria completa (ENDES),0.2896,OEI.02
9,cobertura_juntos_pct,Cobertura del programa Juntos (MIDIS),0.2691,OEI.01 / OEI.03


In [2]:
# 2. Distritos Críticos que activan la Regla Fija 2 (DCI proyectada > 25.0% -> Canal Algedónico S5)
DISTRITOS_CRITICOS_S5 = [
    {'ubigeo': '090411', 'provincia': 'Castrovirreyna', 'distrito': 'Santa Ana', 'dci_actual_pct': 27.50, 'dci_proyectada_pct': 29.29, 'canal_msv': 'S5_ALGEDONICO', 'top_shap': 'DCI previa + Heladas (4 469 msnm)', 'oei_pei': 'OEI.01, OEI.04, OEI.10', 'ods': 'ODS 2, ODS 3, ODS 6'},
    {'ubigeo': '090610', 'provincia': 'Huaytará', 'distrito': 'San Antonio de Cusicancha', 'dci_actual_pct': 26.80, 'dci_proyectada_pct': 27.13, 'canal_msv': 'S5_ALGEDONICO', 'top_shap': 'Baja cloración de agua + Heladas', 'oei_pei': 'OEI.01, OEI.04, OEI.10', 'ods': 'ODS 2, ODS 3, ODS 6'},
    {'ubigeo': '090607', 'provincia': 'Huaytará', 'distrito': 'Pilpichaca', 'dci_actual_pct': 26.50, 'dci_proyectada_pct': 27.04, 'canal_msv': 'S5_ALGEDONICO', 'top_shap': 'Altitud extrema (4 092 msnm) + Heladas', 'oei_pei': 'OEI.01, OEI.04, OEI.10', 'ods': 'ODS 2, ODS 3, ODS 6'},
    {'ubigeo': '090304', 'provincia': 'Angaraes', 'distrito': 'Ccochaccasa', 'dci_actual_pct': 26.20, 'dci_proyectada_pct': 26.16, 'canal_msv': 'S5_ALGEDONICO', 'top_shap': 'Baja cobertura de agua segura + Heladas', 'oei_pei': 'OEI.01, OEI.04, OEI.10', 'ods': 'ODS 2, ODS 3, ODS 6'},
    {'ubigeo': '090102', 'provincia': 'Huancavelica', 'distrito': 'Acobambilla', 'dci_actual_pct': 25.40, 'dci_proyectada_pct': 25.08, 'canal_msv': 'S5_ALGEDONICO', 'top_shap': 'Brecha CRED + Agua rural', 'oei_pei': 'OEI.01, OEI.04, OEI.10', 'ods': 'ODS 2, ODS 3, ODS 6'}
]
print('=== DISTRITOS QUE ACTIVAN EL CANAL ALGEDÓNICO AUTOMATIZADO HACIA S5 (> 25.0%) ===')
pd.DataFrame(DISTRITOS_CRITICOS_S5)

=== DISTRITOS QUE ACTIVAN EL CANAL ALGEDÓNICO AUTOMATIZADO HACIA S5 (> 25.0%) ===


,ubigeo,provincia,distrito,dci_actual_pct,dci_proyectada_pct,canal_msv,top_shap,oei_pei,ods
0,090411,Castrovirreyna,Santa Ana,27.5,29.29,S5_ALGEDONICO,DCI previa + Heladas (4 469 msnm),"OEI.01, OEI.04, OEI.10","ODS 2, ODS 3, ODS 6"
1,090610,Huaytará,San Antonio de Cusicancha,26.8,27.13,S5_ALGEDONICO,Baja cloración de agua + Heladas,"OEI.01, OEI.04, OEI.10","ODS 2, ODS 3, ODS 6"
2,090607,Huaytará,Pilpichaca,26.5,27.04,S5_ALGEDONICO,Altitud extrema (4 092 msnm) + Heladas,"OEI.01, OEI.04, OEI.10","ODS 2, ODS 3, ODS 6"
3,090304,Angaraes,Ccochaccasa,26.2,26.16,S5_ALGEDONICO,Baja cobertura de agua segura + Heladas,"OEI.01, OEI.04, OEI.10","ODS 2, ODS 3, ODS 6"
4,090102,Huancavelica,Acobambilla,25.4,25.08,S5_ALGEDONICO,Brecha CRED + Agua rural,"OEI.01, OEI.04, OEI.10","ODS 2, ODS 3, ODS 6"


In [3]:
# 3. Resumen para el Capítulo 6 del Informe: Coordinación S2 por Red de Salud Provincial (7 Provincias)
print('=== HOMEOSTATO S3-S4 Y COORDINACIÓN S2 POR RED DE SALUD PROVINCIAL ===')
pd.DataFrame.from_dict(METRICS_REPORT['resumen_por_provincia_s2'], orient='index')

=== HOMEOSTATO S3-S4 Y COORDINACIÓN S2 POR RED DE SALUD PROVINCIAL ===


,total_distritos,exactitud_pct,dci_actual_promedio_pct,dci_proyectada_promedio_pct,distritos_alerta_algedonica_s5
Acobamba,8,95.45,18.62,18.17,0
Angaraes,12,93.94,18.70,18.30,1
Castrovirreyna,13,90.21,16.35,15.45,1
Churcampa,11,91.74,16.82,15.95,0
Huancavelica,19,86.60,17.09,16.41,1
Huaytará,16,92.05,17.99,17.34,2
Tayacaja,21,93.51,16.54,15.63,0
